In [0]:
# call Population API
import requests

population_url = (
    "https://honolulu-api.datausa.io/tesseract/data.jsonrecords"
    "?cube=acs_yg_total_population_1"
    "&drilldowns=Year%2CNation"
    "&locale=en"
    "&measures=Population"
)

response = requests.get(
    population_url,
    timeout=60
)

response.raise_for_status()

population_data = response.json()

# Basic validation
if "data" not in population_data:
    raise Exception("Expected 'data' key not found in API response")

print("DataUSA API request successful")
print(f"Records received: {len(population_data['data'])}")
print(f"Top-level keys: {list(population_data.keys())}")

In [0]:
# Write JSON to temporary location
import json

tmp_dir = "/Volumes/bls/bronze/bls_raw/tmp/datausa"
tmp_path = f"{tmp_dir}/population.json"

dbutils.fs.mkdirs(tmp_dir)

with open(tmp_path, "w", encoding="utf-8") as f:
    json.dump(
        population_data,
        f,
        indent=2
    )

# Validate file
tmp_file = dbutils.fs.ls(tmp_path)[0]

if tmp_file.size == 0:
    raise Exception("Population JSON file is empty")

print(f"JSON written successfully: {tmp_path}")
print(f"File size: {tmp_file.size} bytes")

In [0]:
# Move to Raw and verification

raw_dir = "/Volumes/bls/bronze/bls_raw/raw/datausa"
raw_path = f"{raw_dir}/population.json"

# Create raw directory if it doesn't exist
dbutils.fs.mkdirs(raw_dir)

# Remove previous raw copy, if any
dbutils.fs.rm(raw_path, True)

# Move validated file from tmp to raw
dbutils.fs.mv(tmp_path, raw_path)

# Final verification
raw_file = dbutils.fs.ls(raw_path)[0]

print("DataUSA ingestion completed successfully")
print(f"Path: {raw_path}")
print(f"File size: {raw_file.size} bytes")